# Recource Counting

## 引例

- 问题：用1024张H100，在15万亿个token上，训练一个70B参数的模型，需要多长时间？
- 计算：
    - total_flops = 6 * 70e9 * 15e12
    - h100_flop/sec = 1979e12 / 2
    - mfu = 0.5  **?**
    - flops/day = h100_flop/sec * mfu * 1024 *60 *60 * 24
    - days = total_flops * flops/day

- 问题：如果用八块H100配合adamW，能训练的最大模型有多大？
- 计算：
    - h100_bytes = 80e9(80GB)
    - bytes/parameter = 2 + 2 + (4 + 4)
    - num_parameters = h100_bytes * 8 / bytes/parameter
    

本节要点：
- pytorch
- 写代码养成资源核算习惯

## 从头搭建

什么是tensors(张量)：
- 参数，梯度，优化器状态，激活值
- 模型本质是一堆形状和精度各异的张量

**float32**（单精度）
- 1位：符号位
- 8位：指数位



In [24]:
# import torch.nn.functional as F
import torch
from torch import nn
from pathlib import Path
import sys
sys.path.insert(0, r".\lectures-main\lectures-main")


from gpu_util import cuda_if_available, get_max_memory_usage

from einops import rearrange, einsum, reduce


In [4]:
def get_memory_usage(x:torch.Tensor):
    return x.numel() * x.element_size()

In [12]:
x = torch.zeros(4,8)
x.dtype
x.numel() # .numel()方法计算元素总个数
x.element_size() # 浮点数4bytes = 32 bits

get_memory_usage(x)
get_memory_usage(torch.empty(12288 * 4, 12288)) 
# 张量：第0维12288*4，第1维12288，每个元素的比特数是8
# 总内存占用：12288*4*12288*4=2415919104

2415919104

- FP32的指数位数：8
- FP16的指数位数：5（容易出现上下溢出/NaN问题）
    - 所以BFloat16被发明
- 小模型可以随便用，FP32更保险；FP16容易出现问题。BFloat16是权衡选择，或者混合精度训练（一部分计算用一个精度，另一部分计算用另一个精度）
    - 一般BF16用于参数、激活值、梯度，FP32用于优化器状态
- 部分NVDIA的transformer engine支持FP8

# einops

einops:受爱因斯坦运算符号启发，是一个跨张量库的统一操作工具，核心理念是关注输入和输出的形状，而不是操作过程，让代码更加易读

In [19]:
x = torch.ones(4,2,3)
y = torch.ones(4,2,3)
z = x @ y.transpose(-2,-1) # 对y倒数第一个维度和倒数第二个维度进行转置
z

tensor([[[3., 3.],
         [3., 3.]],

        [[3., 3.],
         [3., 3.]],

        [[3., 3.],
         [3., 3.]],

        [[3., 3.],
         [3., 3.]]])